In [1]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import pandas as pd
from sklearn.tree import plot_tree
from sklearn.model_selection import train_test_split
from sksurv.ensemble import RandomSurvivalForest
from sksurv.linear_model import CoxPHSurvivalAnalysis
from sksurv.metrics import concordance_index_censored , concordance_index_ipcw
from sklearn.impute import SimpleImputer
from sksurv.util import Surv

/home/diego/miniconda3/envs/DC/lib/python3.10/site-packages/sksurv/__init__.py:5: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


In [2]:
train_folder_path = "./data/X_train/"
test_folder_path = "./data/X_test/"

# Clinical Data
df = pd.read_csv(train_folder_path + "clinical_train.csv", sep=",")
df_eval = pd.read_csv(test_folder_path + "clinical_test.csv", sep=",")

# Molecular Data
maf_df = pd.read_csv(train_folder_path + "molecular_train.csv", sep=",")
maf_eval = pd.read_csv(test_folder_path + "molecular_test.csv", sep=",")

target_df = pd.read_csv(train_folder_path + "target_train.csv", sep=",")
target_df["OS_YEARS"] = pd.to_numeric(target_df["OS_YEARS"], errors="coerce")
target_df["OS_STATUS"] = target_df["OS_STATUS"].astype(bool)

# Preview the data
target_df.head()

,ID,OS_YEARS,OS_STATUS
0,P132697,1.115068,True
1,P132698,4.928767,False
2,P116889,2.043836,False
3,P132699,2.476712,True
4,P132700,3.145205,False


In [3]:
# Drop rows where 'OS_YEARS' is NaN if conversion caused any issues
target_df.dropna(subset=['OS_YEARS', 'OS_STATUS'], inplace=True)

# Check the data types to ensure 'OS_STATUS' is boolean and 'OS_YEARS' is numeric
print(target_df[['OS_STATUS', 'OS_YEARS']].dtypes)

# Contarget_dfvert 'OS_YEARS' to numeric if it isn’t already
target_df['OS_YEARS'] = pd.to_numeric(target_df['OS_YEARS'], errors='coerce')

# Ensure 'OS_STATUS' is boolean
target_df['OS_STATUS'] = target_df['OS_STATUS'].astype(bool)

OS_STATUS       bool
OS_YEARS     float64
dtype: object


In [12]:
# Step: Extract the number of somatic mutations per patient
# Group by 'ID' and count the number of mutations (rows) per patient
tmp = maf_df.groupby('ID').size().reset_index(name='Nmut')
tmp_eval = maf_eval.groupby('ID').size().reset_index(name='Nmut')

# Merge with the training dataset and replace missing values in 'Nmut' with 0
df = df.merge(tmp, on='ID', how='left').fillna({'Nmut': 0})
df_eval = df_eval.merge(tmp_eval, on='ID', how='left').fillna({'Nmut': 0})

# Make a copy of your mutation dataframe
maf_df_copy = maf_df.copy()
maf_eval_copy = maf_eval.copy()

# Ensure proper datatypes
maf_df_copy["VAF"] = pd.to_numeric(maf_df_copy["VAF"], errors="coerce")
maf_df_copy["ID"] = maf_df_copy["ID"].astype(str)
maf_df_copy["GENE"] = maf_df_copy["GENE"].astype(str)

maf_eval_copy["VAF"] = pd.to_numeric(maf_eval_copy["VAF"], errors="coerce")
maf_eval_copy["ID"] = maf_eval_copy["ID"].astype(str)
maf_eval_copy["GENE"] = maf_eval_copy["GENE"].astype(str)

# Step 1️⃣: Pivot to create a matrix of patients (rows) × genes (columns)
# If a patient has multiple mutations in the same gene, take the max VAF
gene_matrix = (
    maf_df_copy
    .groupby(["ID", "GENE"])["VAF"]
    .sum()                          # sum all VAF per gene per patient
    .unstack(fill_value=0)          # convert to wide format
    .reset_index()
)

gene_eval_matrix = (
    maf_eval_copy
    .groupby(["ID", "GENE"])["VAF"]
    .sum()                          # sum all VAF per gene per patient
    .unstack(fill_value=0)          # convert to wide format
    .reset_index()
)

# Step 2️⃣: Merge with your clinical dataframe
df = df.merge(gene_matrix, on="ID", how="left")
df_eval = df_eval.merge(gene_eval_matrix, on="ID", how="left")

# Step 3️⃣: Replace missing values (patients without a given gene mutation)
df = df.fillna(0)
df_eval = df_eval.fillna(0)

# ✅ Now df_with_genes has one column per gene (e.g. TP53, TET2, DNMT3A, etc.)
# Each value = VAF if mutated, or 0 if not mutated
print(df.shape)
print(df_eval.shape)

(3323, 388)
(1193, 265)


In [13]:
# Step 1: Compute weighted mutation burden per patient
tmp = (
    maf_df_copy
    .assign(weighted_effect = maf_df_copy["START"] * maf_df_copy["VAF"] / 1e6) #
    .groupby("ID", as_index=False)["weighted_effect"]
    .mean()
    .rename(columns={"weighted_effect": "Nmut_4"})
)

tmp_eval = (
    maf_eval_copy
    .assign(weighted_effect = maf_eval_copy["START"] * maf_eval_copy["VAF"] / 1e6) #
    .groupby("ID", as_index=False)["weighted_effect"]
    .mean()
    .rename(columns={"weighted_effect": "Nmut_4"})
)

# Step 2: Merge with the clinical dataset
df = (
    df
    .merge(tmp, on="ID", how="left")
    .fillna({"Nmut_4": 0})  # patients with no mutations get Nmut = 0
)

df_eval = (
    df_eval
    .merge(tmp_eval, on="ID", how="left")
    .fillna({"Nmut_4": 0})  # patients with no mutations get Nmut = 0
)

df.head()

,ID,CENTER,BM_BLAST,WBC,ANC,MONOCYTES,HB,PLT,CYTOGENETICS,Nmut_x,...,TP53,U2AF1,U2AF2,WHSC1,WT1,ZBTB33,ZMYM3,ZNF318,ZRSR2,Nmut_4
0,P132697,MSK,14.0,2.8,0.2,0.7,7.6,119.0,"46,xy,del(20)(q12)[2]/46,xy[18]",9.0,...,0.0,0.35,0.0,0.0,0.000,0.0,0.0,0.0,0.000,11.503142
1,P132698,MSK,1.0,7.4,2.4,0.1,11.6,42.0,"46,xx",3.0,...,0.0,0.00,0.0,0.0,0.000,0.0,0.0,0.0,0.000,6.470014
2,P116889,MSK,15.0,3.7,2.1,0.1,14.2,81.0,"46,xy,t(3;3)(q25;q27)[8]/46,xy[12]",3.0,...,0.0,0.00,0.0,0.0,0.035,0.0,0.0,0.0,0.000,5.283016
3,P132699,MSK,1.0,3.9,1.9,0.1,8.9,77.0,"46,xy,del(3)(q26q27)[15]/46,xy[5]",11.0,...,0.0,0.00,0.0,0.0,0.000,0.0,0.0,0.0,0.049,13.988418
4,P132700,MSK,6.0,128.0,9.7,0.9,11.1,195.0,"46,xx,t(3;9)(p13;q22)[10]/46,xx[10]",1.0,...,0.0,0.00,0.0,0.0,0.000,0.0,0.0,0.0,0.000,14.645694


In [14]:
df_eval.head()

,ID,CENTER,BM_BLAST,WBC,ANC,MONOCYTES,HB,PLT,CYTOGENETICS,Nmut_x,...,TERT,TET2,TP53,U2AF1,U2AF2,USP9X,VEGFA,WT1,ZRSR2,Nmut_4
0,KYW1,KYW,68.0,3.45,0.5865,0.0,7.6,48.0,"47,XY,+X,del(9)(q?)[15]/47,XY,+X[5]",4.0,...,0.0,0.000,0.0,0.000,0.0,0.0,0.0,0.000,0.0,16.607177
1,KYW2,KYW,35.0,3.18,1.2402,0.0,10.0,32.0,"46,XY,der(3)?t(3;11)(q26.2;q23),add(4)(p15).de...",3.0,...,0.0,0.000,0.0,0.000,0.0,0.0,0.0,0.000,0.0,14.439185
2,KYW3,KYW,0.0,12.40,8.6800,0.0,12.3,25.0,"47,XX,+8",3.0,...,0.0,0.327,0.0,0.000,0.0,0.0,0.0,0.000,0.0,13.236150
3,KYW4,KYW,61.0,5.55,2.0535,0.0,8.0,44.0,Normal,3.0,...,0.0,0.000,0.0,0.000,0.0,0.0,0.0,0.424,0.0,14.630739
4,KYW5,KYW,2.0,1.21,0.7381,0.0,8.6,27.0,"43,XY,dic(5;17)(q11.2;p11.2),-7,-13,-20,-22,+r...",3.0,...,0.0,0.000,0.0,0.349,0.0,0.0,0.0,0.000,0.0,13.509655


In [15]:
# Select features
features = ['BM_BLAST', 'HB', 'PLT', 'WBC', 'MONOCYTES', 'Nmut','Nmut_4','TP53',
                  'TET2','SF3B1','CBL','ZRSR2','EZH2','U2AF1']
target = ['OS_YEARS', 'OS_STATUS']

# Create the survival data format
X = df.loc[df['ID'].isin(target_df['ID']), features]
X_eval = df_eval[features].copy()
y = Surv.from_dataframe('OS_STATUS', 'OS_YEARS', target_df)
display(X_eval.head(5))

,BM_BLAST,HB,PLT,WBC,MONOCYTES,Nmut,Nmut_4,TP53,TET2,SF3B1,CBL,ZRSR2,EZH2,U2AF1
0,68.0,7.6,48.0,3.45,0.0,4.0,16.607177,0.0,0.000,0.0,0.0,0.0,0.0,0.000
1,35.0,10.0,32.0,3.18,0.0,3.0,14.439185,0.0,0.000,0.0,0.0,0.0,0.0,0.000
2,0.0,12.3,25.0,12.40,0.0,3.0,13.236150,0.0,0.327,0.0,0.0,0.0,0.0,0.000
3,61.0,8.0,44.0,5.55,0.0,3.0,14.630739,0.0,0.000,0.0,0.0,0.0,0.0,0.000
4,2.0,8.6,27.0,1.21,0.0,3.0,13.509655,0.0,0.000,0.0,0.0,0.0,0.0,0.349


In [16]:
from sksurv.ensemble import RandomSurvivalForest
from time import time

imputer = SimpleImputer(strategy="median")
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=features)

# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(
    X_imputed, y, test_size=0.3, random_state=42
)

start = time()

RSF = RandomSurvivalForest(n_estimators = 150,
                        min_samples_split = 6,
                        min_samples_leaf = 2,
                        max_depth = 9)

model = RSF.fit(X_imputed, y)

In [17]:
X_eval_imputed = pd.DataFrame(imputer.fit_transform(X_eval), columns=features)
pred_eval = model.predict(X_eval_imputed)

In [18]:
# Construire le DataFrame soumis
submission = pd.DataFrame({
    "ID": df_eval['ID'],
    "risk_score": pred_eval
})

# Définir l'index comme demandé
submission = submission.set_index("ID")

# Sauvegarder en CSV
submission.to_csv("predictions_risk_scores_RSF_Nmut4.csv")

submission.head()


,risk_score
ID,
KYW1,1093.958818
KYW2,1047.975722
KYW3,472.946156
KYW4,1077.346796
KYW5,841.373747


In [43]:
from sksurv.ensemble import GradientBoostingSurvivalAnalysis

imputer = SimpleImputer(strategy="median")
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=features)

# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(
    X_imputed, y, test_size=0.3, random_state=42
)

start = time()

GBM = GradientBoostingSurvivalAnalysis(learning_rate = 0.05, 
                           max_depth = 3, 
                           min_samples_split = 2, 
                           n_estimators = 200)

#learning_rate = 0.05, max_depth = 3, min_samples_split = 2, n_estimators = 200

#model = GBM.fit(X_train, y_train)
model = GBM.fit(X_imputed, y)

In [46]:
X_eval_imputed = pd.DataFrame(imputer.fit_transform(X_eval), columns=features)
pred_eval = model.predict(X_eval_imputed)

In [47]:
# Construire le DataFrame soumis
submission = pd.DataFrame({
    "ID": df_eval['ID'],
    "risk_score": pred_eval
})

# Définir l'index comme demandé
submission = submission.set_index("ID")

# Sauvegarder en CSV
submission.to_csv("predictions_risk_scores_GBM.csv")

submission.head()

,risk_score
ID,
KYW1,0.734160
KYW2,0.565024
KYW3,-0.380454
KYW4,0.669671
KYW5,0.333971
